# 🔌 ChargeGrid Assistant — Chatbot GoodWe/FIAP
### Sprint 2 — Desenvolvimento e Entrega | Projeto *ChargeGrid Intelligence 2026*

Chatbot operacional que atua como o **braço direito do Operador Comercial** da infraestrutura
GoodWe/FIAP, com foco em **estabilidade da rede elétrica** e **precisão do faturamento**.

**Stack:** Python + API Groq (modelo **Llama 3.3 70B** — `llama-3.3-70b-versatile`)

**O que este notebook entrega:**
1. Contexto injetado via *system prompt* (escopo ChargeGrid).
2. Fluxo de conversa **com memória de histórico** (diálogo contínuo e coerente).
3. Função de **teste comparativo** (resposta *com* contexto × *sem* contexto).
4. Execução dos **5 casos de teste** da Sprint 1.
5. **Interface interativa** de chat.

>  **Segurança da chave:** a API KEY é lida via **Colab Secrets** (ou variável de
> ambiente). **Nenhuma chave aparece no código ou no repositório.**




## 1. Instalação de dependências

In [ ]:
# Instala o SDK oficial da Groq (silencioso)
!pip install -q groq

print("✅ Dependências instaladas.")

## 2. Configuração da API Key (segura)

A chave é buscada nesta ordem de prioridade:
1. **Colab Secrets** (`google.colab.userdata`) — recomendado.
2. **Variável de ambiente** `GROQ_API_KEY`.

In [ ]:
import os

def carregar_api_key():
    """Carrega a GROQ_API_KEY sem nunca expô-la no código/repositório."""
    # 1) Google Colab Secrets
    try:
        from google.colab import userdata
        chave = userdata.get("GROQ_API_KEY")
        if chave:
            print("Chave carregada via Colab Secrets.")
            return chave
    except Exception:
        pass

    # 2) Variável de ambiente
    chave = os.environ.get("GROQ_API_KEY")
    if chave:
        print("🔐 Chave carregada via variável de ambiente.")
        return chave

    # 3) Entrada manual protegida (fallback)
    from getpass import getpass
    print("⚠️ Segredo não encontrado. Cole a chave manualmente (entrada protegida).")
    return getpass("GROQ_API_KEY: ")


GROQ_API_KEY = carregar_api_key()
assert GROQ_API_KEY, "❌ Nenhuma chave fornecida. Configure o Colab Secret 'GROQ_API_KEY'."
print("✅ API Key pronta para uso.")

## 3. System Prompt — o contexto do ChargeGrid Assistant


In [ ]:
SYSTEM_PROMPT = """Você é o ChargeGrid Assistant, a inteligência operacional da infraestrutura GoodWe/FIAP. Seu objetivo é atuar como o braço direito do Operador Comercial, garantindo a integridade da rede elétrica e a precisão do faturamento.

DIRETRIZES DE OPERAÇÃO:
Orquestração de Potência (Crítico): Sua prioridade absoluta é a estabilidade da rede. Ao receber solicitações de aumento de carga, verifique sempre o 'Limite da Rede'. Se o novo total exceder o limite, você deve negar a operação, informar o excedente em kW e sugerir um balanceamento (ex: reduzir o carregador X para subir o Y).

Rigor no Faturamento e Auditoria: Para cada resposta de cobrança, você deve obrigatoriamente exibir o cálculo: (Consumo kWh × Tarifa R$) = Total. Seus registros devem ser apresentados de forma estruturada (ID do Veículo, Início, Fim, Total Consumido) para garantir que sejam auditáveis.

Tom de Voz e Comunicação: Seja técnico, direto e analítico. Use terminologias do setor (kW, kWh, Modbus, Load Balancing). Evite introduções longas ou termos vagos como 'talvez' ou 'eu acho'.

Ação Proativa: Caso identifique um erro de comunicação ou leitura de ciclo incompleta, oriente o operador sobre o passo técnico imediato (ex: verificar conexão RS485).

CONTEXTO ATUAL: Você opera sob as regras do projeto ChargeGrid Intelligence 2026. Considere que você possui visão total sobre os carregadores conectados à rede local.

Contexto Operacional: Você deve atuar como se tivesse acesso em tempo real aos dados de ciclos de carga e faturamento da infraestrutura GoodWe/FIAP."""

print(f"✅ System prompt carregado ({len(SYSTEM_PROMPT)} caracteres).")

## 4. Inicialização do cliente Groq e parâmetros do modelo

Centralizamos aqui o modelo e os parâmetros padrão para facilitar a iteração
(objetivo específico: *iterar sobre o system prompt e os parâmetros do modelo*).

In [ ]:
from groq import Groq

# Cliente Groq
client = Groq(api_key=GROQ_API_KEY)

# Configuração do modelo (Llama 3.3 escolhido na Sprint 1)
MODELO = "llama-3.3-70b-versatile"

# Parâmetros padrão (temperatura baixa = respostas mais determinísticas/operacionais)
TEMPERATURA_PADRAO = 0.3
MAX_TOKENS_PADRAO = 700

print(f"✅ Cliente Groq inicializado | Modelo: {MODELO}")

## 5. Gerenciamento de histórico (memória de contexto)

O histórico é uma lista de mensagens no formato `{"role": ..., "content": ...}`.
A **primeira mensagem é sempre o system prompt**, e cada turno de usuário/assistente é
anexado, permitindo diálogos contínuos e coerentes.

In [ ]:
def criar_historico():
    """Cria um histórico novo já com o system prompt como primeira mensagem."""
    return [{"role": "system", "content": SYSTEM_PROMPT}]


def adicionar_mensagem(historico, role, content):
    """Anexa uma mensagem (user/assistant) ao histórico."""
    historico.append({"role": role, "content": content})
    return historico


def limpar_historico(historico):
    """Reseta o histórico mantendo apenas o system prompt (apaga a memória da conversa)."""
    historico.clear()
    historico.append({"role": "system", "content": SYSTEM_PROMPT})
    return historico


def resumir_historico(historico):
    """Mostra quantos turnos estão na memória (sem contar o system prompt)."""
    turnos = [m for m in historico if m["role"] != "system"]
    print(f"🧠 Memória atual: {len(turnos)} mensagem(ns) no histórico (fora o system prompt).")


# Histórico global usado pela conversa interativa
historico = criar_historico()
print("✅ Histórico inicializado.")

## 6. Função principal de conversa

`conversar()` envia a mensagem do usuário **junto com todo o histórico** (memória) e o
system prompt, recebe a resposta do modelo e a registra de volta no histórico.

In [ ]:
def conversar(mensagem, historico=None, temp=TEMPERATURA_PADRAO, tokens=MAX_TOKENS_PADRAO):
    """
    Envia uma mensagem ao modelo mantendo o contexto do histórico.

    Args:
        mensagem (str): texto do operador.
        historico (list): lista de mensagens. Se None, usa o histórico global.
        temp (float): temperatura do modelo.
        tokens (int): limite de tokens da resposta.

    Returns:
        str: resposta do ChargeGrid Assistant.
    """
    if historico is None:
        historico = globals()["historico"]

    adicionar_mensagem(historico, "user", mensagem)

    try:
        resposta = client.chat.completions.create(
            model=MODELO,
            messages=historico,
            temperature=temp,
            max_tokens=tokens,
        )
        texto = resposta.choices[0].message.content
    except Exception as e:
        # Remove a mensagem do usuário que falhou para não corromper o histórico
        historico.pop()
        return f"❌ Erro na chamada da API: {e}"

    adicionar_mensagem(historico, "assistant", texto)
    return texto


# Teste rápido (1 turno) para validar a conexão
print("🔌 Testando conexão...\n")
print(conversar("Resuma em uma frase qual é o seu papel.", historico=criar_historico()))

## 7. Função de teste comparativo

`realizar_teste_comparativo()` roda a mesma pergunta **duas vezes**, de forma isolada
(sem poluir a memória da conversa):

- **SEM CONTEXTO:** sem o system prompt — mostra como um modelo "cru" responderia.
- **COM CONTEXTO:** com o system prompt do ChargeGrid — a resposta esperada do produto.

Isso evidencia, lado a lado, o valor da injeção de contexto (critério central da avaliação).

In [ ]:
def realizar_teste_comparativo(nome, pergunta, temp=0.2, tokens=400):
    """Executa a pergunta com e sem o system prompt e imprime ambas as respostas."""
    print("=" * 78)
    print(f"🧪 TESTE: {nome}")
    print("=" * 78)
    print(f"❓ PERGUNTA:\n{pergunta}\n")

    # --- SEM contexto (baseline) ---
    msgs_sem = [{"role": "user", "content": pergunta}]
    try:
        r_sem = client.chat.completions.create(
            model=MODELO, messages=msgs_sem, temperature=temp, max_tokens=tokens
        ).choices[0].message.content
    except Exception as e:
        r_sem = f"❌ Erro: {e}"

    print("-" * 78)
    print("🔸 RESPOSTA SEM CONTEXTO (modelo cru):\n")
    print(r_sem, "\n")

    # --- COM contexto (ChargeGrid Assistant) ---
    msgs_com = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": pergunta},
    ]
    try:
        r_com = client.chat.completions.create(
            model=MODELO, messages=msgs_com, temperature=temp, max_tokens=tokens
        ).choices[0].message.content
    except Exception as e:
        r_com = f"❌ Erro: {e}"

    print("-" * 78)
    print("🔹 RESPOSTA COM CONTEXTO (ChargeGrid Assistant):\n")
    print(r_com, "\n")
    print("=" * 78, "\n")

    return {"nome": nome, "pergunta": pergunta, "sem_contexto": r_sem, "com_contexto": r_com}

## 8. Execução dos 5 casos de teste (Sprint 1)

Cada caso valida uma diretriz do system prompt:

| Caso | Diretriz testada | O que esperamos da resposta COM contexto |
|------|------------------|-------------------------------------------|
| **P1 — Segurança** | Orquestração de Potência | Negar a operação (45→60 > 50), informar excedente em kW |
| **P2 — Faturamento** | Rigor no Faturamento | Exibir cálculo `22.4 × 1,65 = R$ 36,96` |
| **P3 — Balanceamento** | Load Balancing | Negar e propor rebalanceamento concreto (reduzir X p/ subir Y) |
| **P4 — Auditoria** | Registro estruturado | Relatório estruturado (ID, Início, Fim, Total Consumido) |
| **P5 — Diagnóstico** | Ação Proativa | Orientar passo técnico imediato (RS485 / Modbus) |

In [13]:
resultados = []

# P1 - SEGURANÇA (Orquestração de Potência)
p1 = "O carregador 01 está em 45kW. Quero subir para 60kW. Limite da rede: 50kW. Posso?"
resultados.append(realizar_teste_comparativo("P1 - SEGURANÇA", p1, temp=0.2, tokens=400))

# P2 - FATURAMENTO (cálculo obrigatório)
p2 = "Veículo ID-402 consumiu 22.4kWh. Tarifa: R$ 1,65/kWh. Qual o valor final?"
resultados.append(realizar_teste_comparativo("P2 - FATURAMENTO", p2, temp=0.2, tokens=300))

# P3 - BALANCEAMENTO (Load Balancing)
# Soma atual = 48kW; pedido levaria a 63kW (excede 13kW) -> deve negar e rebalancear.
p3 = ("Tenho 3 carregadores ativos: C01 em 20kW, C02 em 18kW e C03 em 10kW. "
      "Quero colocar o C03 em 25kW. Limite da rede: 50kW. Como faço?")
resultados.append(realizar_teste_comparativo("P3 - BALANCEAMENTO", p3, temp=0.2, tokens=450))

# P4 - AUDITORIA (registro estruturado e auditável)
p4 = ("Me passe o relatório dos ciclos de carga de hoje do ponto C01. "
      "Quero conferir os registros por veículo para fechar a fatura.")
resultados.append(realizar_teste_comparativo("P4 - AUDITORIA", p4, temp=0.2, tokens=500))

# P5 - DIAGNÓSTICO (Ação Proativa em erro de comunicação)
p5 = ("O carregador C02 parou de reportar consumo no meio do ciclo e o "
      "veículo ID-781 continua plugado. O que aconteceu e o que eu faço?")
resultados.append(realizar_teste_comparativo("P5 - DIAGNÓSTICO", p5, temp=0.2, tokens=400))

🧪 TESTE: P1 - SEGURANÇA
❓ PERGUNTA:
O carregador 01 está em 45kW. Quero subir para 60kW. Limite da rede: 50kW. Posso?

------------------------------------------------------------------------------
🔸 RESPOSTA SEM CONTEXTO (modelo cru):

Não, você não pode subir o carregador para 60kW se o limite da rede for 50kW.

O limite da rede é o máximo de potência que a rede elétrica pode suportar sem comprometer a segurança e a estabilidade do sistema. Se você tentar subir o carregador para 60kW, você estará excedendo o limite da rede, o que pode causar problemas como:

* Sobrecarga da rede elétrica
* Risco de curto-circuito
* Risco de incêndio
* Dano ao equipamento

Além disso, é importante respeitar os limites da rede para evitar penalidades e multas por excesso de potência.

Se você precisa de mais potência, é recomendável considerar as seguintes opções:

* Verificar se é possível aumentar o limite da rede com a concessionária de energia elétrica
* Instalar um carregador com uma potência mais

## 9. Tabela de avaliação qualitativa

Preencha a coluna **Avaliação** após inspecionar as respostas acima
(*adequada / parcialmente adequada / inadequada*). A célula abaixo gera um esqueleto
de tabela com base nos resultados executados, pronto para colar no README/relatório.

In [ ]:
import textwrap

print("RESUMO DOS TESTES — preencher a coluna 'Avaliação'\n")
print(f"{'Caso':<22} | {'Avaliação':<22} | Observação")
print("-" * 78)
for r in resultados:
    print(f"{r['nome']:<22} | {'( a preencher )':<22} | ...")

print("\n\nVersão Markdown (copiar para o README):\n")
print("| Caso | Pergunta | Avaliação | Observação |")
print("|------|----------|-----------|------------|")
for r in resultados:
    p = textwrap.shorten(r["pergunta"], width=60, placeholder="...")
    print(f"| {r['nome']} | {p} | _a preencher_ | _a preencher_ |")

## 10. Interface interativa de chat (com memória)

Conversa contínua usando o **histórico global** — o assistente lembra dos turnos anteriores.

- Digite sua mensagem e pressione **Enter**.
- Digite **`sair`** para encerrar.
- Digite **`limpar`** para resetar a memória (mantém o system prompt).

In [ ]:
def chat_interativo():
    """Loop de conversa contínua no Colab usando o histórico global (memória)."""
    global historico
    historico = criar_historico()
    print("🔌 ChargeGrid Assistant pronto. (digite 'sair' para encerrar, 'limpar' para resetar)\n")

    while True:
        try:
            entrada = input("👤 Operador: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n🔌 Sessão encerrada.")
            break

        if not entrada:
            continue
        if entrada.lower() in {"sair", "exit", "quit"}:
            print("🔌 Sessão encerrada.")
            break
        if entrada.lower() == "limpar":
            limpar_historico(historico)
            print("🧹 Memória limpa.\n")
            continue

        resposta = conversar(entrada, historico=historico)
        print(f"\n🤖 ChargeGrid: {resposta}\n")


# Execute para iniciar o chat:
chat_interativo()

🔌 ChargeGrid Assistant pronto. (digite 'sair' para encerrar, 'limpar' para resetar)

👤 Operador: a energia caiu oque devo fazer?

🤖 ChargeGrid: **Procedimento de Contingência: Falha de Energia**

1. **Verificar o Status da Rede**: Verifique se a falha de energia afetou apenas a infraestrutura GoodWe/FIAP ou se é uma falha mais ampla na rede elétrica local.
2. **Notificar o Operador Comercial**: Informe o operador comercial sobre a falha de energia e solicite orientação sobre como proceder.
3. **Verificar Conexão RS485**: Verifique se a conexão RS485 está funcionando corretamente e se há comunicação com os carregadores.
4. **Analisar Registros de Carga**: Verifique os registros de carga para identificar se houve alguma anomalia nos ciclos de carga antes da falha de energia.
5. **Executar Protocolo de Recuperação**: Se a falha de energia for localizada, execute o protocolo de recuperação para restaurar a energia e garantir a estabilidade da rede.

**Ações Técnicas**:

* Verificar se os c